# Evaluación MLP y deriva de covariables

Usá esta notebook para comparar un Champion y un Challenger sobre el **mismo holdout humano congelado**, y para revisar si dos cohortes compatibles de 19 features cambiaron. No entrena, no modifica datasets, no publica bundles, no cambia DVC y no escribe en PostgreSQL.

**Inicio rápido recomendado:** dejá `RUN_MODEL_EVALUATION = False` y `RUN_DRIFT_ANALYSIS = False`, ejecutá `Run All` para validar el entorno, y recién después declarás las rutas exactas que querés auditar.

<details>
<summary><strong>Qué evidencia produce y qué no</strong></summary>

- La comparación MLP usa sólo `Normal`, `Reduced` y `Congested`. `Accident` continúa siendo una confirmación humana: una alerta automática se conserva como `Congested`.
- Las matrices y métricas aplican la cadena completa de serving de cada bundle: scaler, MLP, calibración, umbrales e histéresis.
- El drift compara cohorts `traffic-features-v2`; un CSV legacy no se presenta como drift de las 19 features.
- Los intervalos bootstrap son evidencia descriptiva. La promoción permanece manual y usa los gates ya sellados en el manifiesto.
- Para PostgreSQL se exige el perfil `training`, un intervalo UTC semiabierto y filtros declarados; la consulta es sólo de lectura.

</details>


In [ ]:
# Environment setup — run once per Colab runtime
import importlib.metadata
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/zgfnicolas/vaaet.git"
REPO_DIR = Path("/content/vaaet")
if IN_COLAB:
    if (REPO_DIR / ".git").is_dir():
        subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
    else:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
    REPO_ROOT = REPO_DIR.resolve()
else:
    REPO_ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file() and (path / "src/vaaet").is_dir()), None)
    if REPO_ROOT is None:
        raise RuntimeError("No se encontró la raíz del repositorio VAAET.")
os.chdir(REPO_ROOT)

def validate_runtime_version(version: tuple[int, int]) -> None:
    if not (3, 10) <= version <= (3, 13):
        raise RuntimeError(f"Unsupported Python {version[0]}.{version[1]}. VAAET supports Python 3.10–3.13.")

def install_project(command: list[str], *, extras: str) -> None:
    result = subprocess.run(command, capture_output=True, text=True, check=False)
    if result.returncode == 0:
        print(f"✅ Instalación de VAAET terminada | extras={extras}")
        return
    print("----- pip stdout -----")
    print(result.stdout.strip() or "(empty)")
    print("----- pip stderr -----")
    print(result.stderr.strip() or "(empty)")
    raise RuntimeError(f"VAAET installation failed under Python {sys.version.split()[0]} with extras={extras}. Use runtime 2026.07 or update the repository.")

validate_runtime_version((sys.version_info.major, sys.version_info.minor))
WORKFLOW_EXTRAS = "training,visualization,database"
project_requirement = f"{REPO_ROOT}[{WORKFLOW_EXTRAS}]"
install_command = [sys.executable, "-m", "pip", "install", "-q"]
if IN_COLAB:
    install_command.append(project_requirement)
else:
    install_command.extend(["-e", project_requirement])
install_project(install_command, extras=WORKFLOW_EXTRAS)
for module_name in tuple(sys.modules):
    if module_name == "vaaet" or module_name.startswith("vaaet."):
        sys.modules.pop(module_name, None)
importlib.invalidate_caches()

import vaaet

def validate_vaaet_origin(package: object, repo_root: Path, in_colab: bool) -> Path:
    package_file = getattr(package, "__file__", None)
    if not package_file:
        raise ImportError("The 'vaaet' import resolved to a namespace package. Re-run this setup cell.")
    origin = Path(package_file).resolve()
    if in_colab and repo_root.resolve() in origin.parents:
        raise ImportError(f"Colab debe cargar el paquete instalado, no el repositorio: {origin}")
    if not in_colab and origin.parent != (repo_root / "src/vaaet").resolve():
        raise ImportError(f"La instalación editable local tiene un origen inesperado: {origin}")
    return origin

VAAET_PACKAGE_FILE = validate_vaaet_origin(vaaet, REPO_ROOT, IN_COLAB)
pip_check = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True, check=False)
pip_check_output = "\n".join(part.strip() for part in (pip_check.stdout, pip_check.stderr) if part.strip())
if pip_check.returncode:
    print("⚠️ pip check detectó conflictos administrados; los imports del flujo se validan a continuación.")
    print(pip_check_output or "Sin diagnóstico de pip.")
else:
    print("✅ pip check: dependencias consistentes")

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg2
import sqlalchemy
import tensorflow as tf

from vaaet.data.database import DatabaseProfile, get_optional_database_settings, load_telemetry_window
from vaaet.evaluation.champion_challenger import evaluate_champion_challenger, load_evaluation_bundle, plot_champion_challenger_confusion
from vaaet.evaluation.drift import build_feature_cohort, build_feature_cohort_from_raw_telemetry, compare_feature_cohorts, plot_feature_drift
from vaaet.training.holdout import FileSystemHoldoutStore

print(f"✅ Evaluación lista | Python {sys.version.split()[0]} | TensorFlow {tf.__version__}")
print(f"Paquete: {VAAET_PACKAGE_FILE}")


In [ ]:
# Workflow configuration — edit only this cell
RUN_MODEL_EVALUATION = False
CHAMPION_BUNDLE_DIR = ""
CHALLENGER_BUNDLE_DIR = ""
HOLDOUT_SNAPSHOT_PATH = ""  # Exact human-holdout-*.zip; never current.json.
BOOTSTRAP_SAMPLES = 1_000

RUN_DRIFT_ANALYSIS = False
REFERENCE_FEATURE_COHORT_PATH = ""  # CSV/ZIP with feature_schema_version + 19 features.
OPERATIONAL_FEATURE_COHORT_PATH = ""
USE_POSTGRES_OPERATIONAL = False
POSTGRES_START_UTC = ""  # Inclusive ISO-8601 timestamp with timezone.
POSTGRES_END_UTC = ""  # Exclusive ISO-8601 timestamp with timezone.
POSTGRES_PIPELINE_RUN_IDS: tuple[str, ...] = ()
POSTGRES_CLIP_IDS: tuple[str, ...] = ()
DRIFT_PLOT_FEATURES: tuple[str, ...] = ()

if RUN_MODEL_EVALUATION:
    required_paths = (CHAMPION_BUNDLE_DIR, CHALLENGER_BUNDLE_DIR, HOLDOUT_SNAPSHOT_PATH)
    if not all(required_paths):
        raise ValueError("La comparación requiere Champion, Challenger y un ZIP de holdout exacto.")
    if Path(HOLDOUT_SNAPSHOT_PATH).name == "current.json":
        raise ValueError("Indicá el human-holdout-*.zip exacto; current.json no es un benchmark reproducible.")
    if BOOTSTRAP_SAMPLES < 1:
        raise ValueError("BOOTSTRAP_SAMPLES debe ser positivo.")
if RUN_DRIFT_ANALYSIS:
    if not REFERENCE_FEATURE_COHORT_PATH:
        raise ValueError("El drift requiere una cohorte de referencia explícita.")
    if bool(OPERATIONAL_FEATURE_COHORT_PATH) == USE_POSTGRES_OPERATIONAL:
        raise ValueError("Elegí exactamente una cohorte operacional: archivo o PostgreSQL read-only.")
    if USE_POSTGRES_OPERATIONAL and (not POSTGRES_START_UTC or not POSTGRES_END_UTC):
        raise ValueError("PostgreSQL requiere inicio y fin UTC explícitos.")
print("✅ Configuración validada | sin lecturas ni escrituras mientras los análisis sigan desactivados")


## 1. Champion vs. Challenger

La notebook valida ambos manifiestos antes de cargar Keras o joblib. Si los fingerprints de holdout no coinciden, la comparación se detiene: benchmarks distintos no son A/B comparables.


In [ ]:
# Cell 1 — Exact frozen-holdout comparison
comparison = None
holdout_snapshot = None
if RUN_MODEL_EVALUATION:
    holdout_path = Path(HOLDOUT_SNAPSHOT_PATH).expanduser().resolve()
    if not holdout_path.is_file():
        raise FileNotFoundError(f"No existe el holdout configurado: {holdout_path.name}")
    holdout_snapshot = FileSystemHoldoutStore(holdout_path.parent).load_snapshot(holdout_path)
    champion_bundle = load_evaluation_bundle(CHAMPION_BUNDLE_DIR, name="Champion")
    challenger_bundle = load_evaluation_bundle(CHALLENGER_BUNDLE_DIR, name="Challenger")
    comparison = evaluate_champion_challenger(
        champion_bundle, challenger_bundle, holdout_snapshot, bootstrap_samples=BOOTSTRAP_SAMPLES
    )
    print(f"✅ Holdout exacto: {holdout_snapshot.descriptor['fingerprint']}")
    for bundle in (champion_bundle, challenger_bundle):
        blockers = bundle.manifest["data_provenance"].get("promotion_blockers", [])
        print(f"{bundle.name} | blockers declarados: {blockers or 'ninguno'}")
    print(comparison.summary.to_string(index=False))
    print("\nIntervalos bootstrap emparejados (95%):")
    print(comparison.bootstrap_intervals.to_string(index=False))
else:
    print("ℹ️ Comparación MLP desactivada; no se cargaron bundles ni holdouts.")


In [ ]:
# Cell 1b — Human-review visual evidence
if comparison is not None and holdout_snapshot is not None:
    plot_champion_challenger_confusion(comparison, holdout_snapshot.test["traffic_state"].to_numpy())
    print("\nSoporte e intervalos del Champion:")
    print(comparison.champion.support_table.to_string(index=False))
    print("\nSoporte e intervalos del Challenger:")
    print(comparison.challenger.support_table.to_string(index=False))


## 2. Deriva de covariables

Pregunta: ¿la cohorte operacional presenta distribuciones o calidad de las 19 features materialmente distintas de la referencia? Los valores se describen; no se convierten en un gatillo automático de reentrenamiento.


In [ ]:
# Cell 2 — Explicit feature cohorts, optionally bounded PostgreSQL raw telemetry
drift_report = None
if RUN_DRIFT_ANALYSIS:
    reference_frame = pd.read_csv(REFERENCE_FEATURE_COHORT_PATH)
    reference_cohort = build_feature_cohort(reference_frame, name="Referencia")
    if USE_POSTGRES_OPERATIONAL:
        database_settings = get_optional_database_settings(DatabaseProfile.TRAINING)
        if database_settings is None:
            raise RuntimeError("PostgreSQL está habilitado, pero falta el perfil training read-only.")
        raw_operational = load_telemetry_window(
            start=pd.Timestamp(POSTGRES_START_UTC), end=pd.Timestamp(POSTGRES_END_UTC),
            pipeline_run_ids=POSTGRES_PIPELINE_RUN_IDS, clip_ids=POSTGRES_CLIP_IDS, settings=database_settings,
        )
        operational_cohort = build_feature_cohort_from_raw_telemetry(raw_operational, name="Operacional PostgreSQL")
    else:
        operational_frame = pd.read_csv(OPERATIONAL_FEATURE_COHORT_PATH)
        operational_cohort = build_feature_cohort(operational_frame, name="Operacional archivo")
    drift_report = compare_feature_cohorts(reference_cohort, operational_cohort)
    print(f"✅ Referencia: {reference_cohort.profile.records} filas / {reference_cohort.profile.clips} clips")
    print(f"✅ Operacional: {operational_cohort.profile.records} filas / {operational_cohort.profile.clips} clips")
else:
    print("ℹ️ Drift desactivado; no se leyó CSV, ZIP ni PostgreSQL.")


In [ ]:
# Cell 2b — Bounded drift story for manual follow-up
if drift_report is not None:
    print(drift_report.summary.to_string(index=False))
    selected_features = DRIFT_PLOT_FEATURES or None
    plot_feature_drift(drift_report, features=selected_features, max_features=6)
    print("Interpretación: PSI y cambios de cuantiles priorizan revisión humana; no modifican features, umbrales ni el MLP.")


## 3. Cierre humano

Revisá primero la compatibilidad del holdout, los blockers de los manifiestos, la incertidumbre de los deltas y la procedencia de las cohortes. Si el Challenger parece mejor, la promoción sigue siendo una decisión humana separada; esta notebook no copia `.keras`, no altera punteros y no integra la Web App.

YOLO y tracking no se evalúan aquí. Ese flujo necesita cajas, clases e identidades anotadas por humanos para medir detección y seguimiento de forma válida.


In [ ]:
# Cell 3 — Read-only completion
print("✅ Evaluación finalizada sin persistencia, promoción ni modificaciones de datos.")
if comparison is not None:
    print("➡️ Siguiente paso humano: revisar métricas, intervalos y blockers de ambos manifiestos.")
if drift_report is not None:
    print("➡️ Siguiente paso humano: investigar las features priorizadas con su cohorte, clips e intervalo declarados.")
